**Eurovision Voting Blocs Across Eras (1975–2024)**

# 01 - Data Acquisition

**Objectives**
- Download the three public data sources used in the project
- Convert the EurovisionDataset JSON file to csv
- Keep a local copy of every raw file in `data/raw/`

**Output:** `data/raw/votes_spijkervet.csv`, `data/raw/contestants_spijkervet.csv`, `data/raw/votes_josago.csv`, `data/raw/dist_cepii.dta`

> Run this notebook from the `src/` folder. Files that already exist are not downloaded again.

In [2]:
import os

import requests
import pandas as pd

from helper_functions import download_and_convert_josago, missing_table, missing_jury_voters, zero_share

## 1. Data Sources

| Source | What it contains | Years used | File |
|---|---|---|---|
| **Spijkervet & Burgoyne, *Eurovision Song Contest Dataset*** (ISMIR 2023, Zenodo 10088892) | Every vote given by every country, per round | 1975–2023 | `votes_spijkervet.csv`, `contestants_spijkervet.csv` |
| **josago97, *EurovisionDataset*** (GitHub) | Same kind of vote records | 2024 only | `votes_josago.csv` |
| **CEPII GeoDist** (Mayer & Zignago, 2011) | Geographic and cultural indicators for every pair of countries | — | `dist_cepii.dta` |

**What is a "vote record"?** In the Grand Final every participating country (the *voter*) gives points to other countries (the *performers*): 12 to its favourite, then 10, 8, 7, 6, 5, 4, 3, 2 and 1. One row of the vote table is one pair *voter → performer* in one round of one year, with the points given. Since 2016 each country gives two separate sets of points, one from a professional jury and one from the public (televote), so the table has three point columns: `total_points`, `jury_points` and `tele_points`.

**Terminology:** in this project a *tie* is a link between two countries in a network, and a *draw* is when two countries receive the same number of points.

The main dataset stops in 2023, so the 2024 contest is taken from EurovisionDataset. `contestants_spijkervet.csv` (songs, artists, running order) is downloaded for reference but is not used in the network pipeline.

In [3]:
DATA_DIR = os.path.join("..", "data", "raw")
os.makedirs(DATA_DIR, exist_ok=True)

SPIJKERVET_BASE = "https://github.com/Spijkervet/eurovision-dataset/releases/download/2023"
JOSAGO_JSON_URL = "https://raw.githubusercontent.com/josago97/EurovisionDataset/main/dataset/eurovision.json"
CEPII_URL = "http://www.cepii.fr/distance/dist_cepii.dta"

## 2. Download Helper

A simple function that streams a file from a URL to disk.

In [4]:
def download_file(url: str, dest_path: str):
    print(f"Downloading {url} -> {dest_path}...")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    resp = requests.get(url, headers=headers, stream=True, timeout=60)
    resp.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f"Successfully saved {dest_path} ({os.path.getsize(dest_path)} bytes)")

## 3. Spijkervet Eurovision Dataset (1975–2023)

### 3.1. Votes

In [5]:
dest_votes_spijkervet = os.path.join(DATA_DIR, "votes_spijkervet.csv")
if not os.path.exists(dest_votes_spijkervet):
    download_file(f"{SPIJKERVET_BASE}/votes.csv", dest_votes_spijkervet)
else:
    print(f"Already exists: {dest_votes_spijkervet}")

Already exists: ..\data\raw\votes_spijkervet.csv


### 3.2. Contestants

In [6]:
dest_contestants_spijkervet = os.path.join(DATA_DIR, "contestants_spijkervet.csv")
if not os.path.exists(dest_contestants_spijkervet):
    download_file(f"{SPIJKERVET_BASE}/contestants.csv", dest_contestants_spijkervet)
else:
    print(f"Already exists: {dest_contestants_spijkervet}")

Already exists: ..\data\raw\contestants_spijkervet.csv


## 4. EurovisionDataset (2024)

This source is a single JSON file with a nested structure:

```
contest (one per year)
└── rounds (semi-finals, final)
    └── performances (one per competing country)
        └── scores: "total", "public" (televote), "jury"
            └── votes: {voter country: points}
```

To use it together with the main dataset, the nested structure is flattened: every *(voter, performer)* pair becomes one row, with the same columns as `votes_spijkervet.csv`.

In [7]:
dest_votes_josago = os.path.join(DATA_DIR, "votes_josago.csv")
if not os.path.exists(dest_votes_josago):
    download_and_convert_josago(JOSAGO_JSON_URL, dest_votes_josago)
else:
    print(f"Already exists: {dest_votes_josago}")

Already exists: ..\data\raw\votes_josago.csv


## 5. CEPII GeoDist

GeoDist describes every ordered pair of countries *(origin, destination)*. Two of its binary variables are used later as country-pair attributes:

- `contig` = 1 if the two countries share a land border;
- `comlang_off` = 1 if they share an official language.

The file Stata format (`.dta`) is preseved.

In [8]:
dest_dist_cepii = os.path.join(DATA_DIR, "dist_cepii.dta")
if not os.path.exists(dest_dist_cepii):
    download_file(CEPII_URL, dest_dist_cepii)
else:
    print(f"Already exists: {dest_dist_cepii}")

Already exists: ..\data\raw\dist_cepii.dta


## 6. Quick Look at the Raw Files

A first check that the files are readable and have the expected columns.

### 6.1. Spijkervet

In [9]:
votes_spijkervet = pd.read_csv(dest_votes_spijkervet)
period_spijkervet = pd.cut(votes_spijkervet["year"], [1956, 2015, 2023], labels=["1957–2015", "2016–2023"])
print(f"Votes: {votes_spijkervet.shape}, years {votes_spijkervet['year'].min()}–{votes_spijkervet['year'].max()}")

Votes: (51354, 9), years 1957–2023


In [10]:
votes_spijkervet.head()

,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
0,1957,final,at,nl,at,nl,6,NaN,NaN
1,1957,final,at,fr,at,fr,0,NaN,NaN
2,1957,final,at,dk,at,dk,0,NaN,NaN
3,1957,final,at,lu,at,lu,3,NaN,NaN
4,1957,final,at,de,at,de,0,NaN,NaN


In [11]:
votes_spijkervet.tail()

,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
51349,2023,final,wld,al,wld,al,6,6.0,NaN
51350,2023,final,wld,pt,wld,pt,0,0.0,NaN
51351,2023,final,wld,rs,wld,rs,0,0.0,NaN
51352,2023,final,wld,gb,wld,gb,0,0.0,NaN
51353,2023,final,wld,de,wld,de,0,0.0,NaN


**Overview:** the file starts in 1957, earlier than the period used in the project (1975 onwards). Country codes are lowercase two-letter codes, plus `wld` for the *Rest of the World* televote (2023).

In [12]:
# looking for missing values
missing_table(votes_spijkervet, period_spijkervet)

rows  total_points  jury_points  tele_points
period    round                                                      
1957–2015 final         30136             0        30136        30136
          semi-final     3751             0         3751         3751
          semi-final-1   2804             0         2804         2804
          semi-final-2   2922             0         2922         2922
2016–2023 final          7370             0           26            0
          semi-final-1   2141             0            0            0
          semi-final-2   2230             0            0            0

In [13]:
# looking for voters without jury points
missing_jury_voters(votes_spijkervet)

year  from_country
2023  wld             26
dtype: int64

**Missing values:** only `jury_points` and `tele_points` have missing values; `total_points` is always filled.

- **1957–2015:** both columns are empty on every row. Before 2016 each country gave a single set of points, so only `total_points` exists.
- **2016–2023:** both columns are filled, except for the 26 Grand Final rows of `wld` in 2023. This online televote has no jury, so `jury_points` is empty.

In both cases an empty value means *"this type of vote did not exist"*, not *"0 points"*, so it is not replaced by zero.

In [14]:
# looking for points = 0
zero_share(votes_spijkervet, period_spijkervet)

period
1957–2015    0.570713
2016–2023    0.437449
Name: total_points, dtype: float64

**Rows with 0 points:** about half of the Grand Final rows have `total_points = 0`. They are kept on purpose: they record that a voter *could* have voted for a performer but did not. Notebook 02 needs them to count in how many years each pair of countries met in the Grand Final. The share is lower from 2016 onwards because each country gives two sets of points (jury and televote), so more performers receive at least one point.

### 6.2. josago

In [15]:
votes_josago = pd.read_csv(dest_votes_josago)
votes_josago_2024 = votes_josago[votes_josago["year"] == 2024]
period_josago = votes_josago_2024["year"].astype(str)
print(f"Votes: {votes_josago.shape}, years {votes_josago['year'].min()}–{votes_josago['year'].max()}")

Votes: (51452, 9), years 1957–2024


In [16]:
votes_josago_2024.head()

,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
49953,2024,final,es,ch,ES,CH,18,6.0,12.0
49954,2024,final,mt,ch,MT,CH,18,6.0,12.0
49955,2024,final,ee,ch,EE,CH,19,7.0,12.0
49956,2024,final,cz,ch,CZ,CH,17,7.0,10.0
49957,2024,final,is,ch,IS,CH,12,6.0,6.0


In [17]:
votes_josago_2024.tail()

,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
51447,2024,semifinal2,al,mt,AL,MT,0,NaN,NaN
51448,2024,semifinal2,fr,mt,FR,MT,1,NaN,NaN
51449,2024,semifinal2,ge,mt,GE,MT,0,NaN,NaN
51450,2024,semifinal2,lv,mt,LV,MT,0,NaN,NaN
51451,2024,semifinal2,sm,mt,SM,MT,4,NaN,NaN


**Overview:** the file covers 1957–2024, but only the 2024 rows are used. Two differences from 6.1: `from_country` and `to_country` are uppercase (`ES`) instead of lowercase (`es`), and the semi-finals are named `semifinal1` / `semifinal2` instead of `semi-final-1` / `semi-final-2`.

In [18]:
# looking for missing values
missing_table(votes_josago_2024, period_josago)

rows  total_points  jury_points  tele_points
period round                                                   
2024   final        925             0           25            0
       semifinal1   270             0          270          270
       semifinal2   304             0          304          304

In [19]:
# looking for voters without jury points
missing_jury_voters(votes_josago_2024)

year  from_country
2024  WLD             25
dtype: int64

**Missing values:**

- **Grand Final:** same as 2023 in 6.1. Only `WLD` (*Rest of the World*) has no `jury_points`.
- **Semi-finals:** new case. `jury_points` and `tele_points` are both empty on every row. Since 2023 the semi-finals are decided by televote only, and the source gives only the total. The semi-finals are not used in the networks.

In [20]:
# looking for points = 0
zero_share(votes_josago_2024, period_josago)

period
2024    0.445405
Name: total_points, dtype: float64

**Rows with 0 points:** the share (44.5%) is close to Spijkervet's 2016–2023 value (43.7%), as expected for the same voting system.

### 6.3. CEPII GeoDist

In [21]:
dist_cepii = pd.read_stata(dest_dist_cepii)
print(f"CEPII GeoDist: {dist_cepii.shape}, {dist_cepii['iso_o'].nunique()} countries")
dist_cepii.head()

CEPII GeoDist: (50176, 14), 224 countries


,iso_o,iso_d,contig,comlang_off,comlang_ethno,colony,comcol,curcol,col45,smctry,dist,distcap,distw,distwces
0,ABW,ABW,0,0,0,0,0,0,0,0,5.225315,5.225315,25.093539,23.047228
1,ABW,AFG,0,0,0,0,0,0,0,0,13257.814453,13257.814453,13168.223762,13166.366934
2,ABW,AGO,0,0,0,0,0,0,0,0,9516.913086,9516.913086,9587.315563,9584.193415
3,ABW,AIA,0,0,1,0,0,0,0,0,983.268250,983.268250,976.897355,976.891580
4,ABW,ALB,0,0,0,0,0,0,0,0,9091.742188,9091.742188,9091.575727,9091.465815


**Overview:** one row per ordered pair *(origin, destination)*, identified by ISO3 codes (`iso_o`, `iso_d`). The pairs are directed, so every pair of countries appears twice (A→B and B→A), and each country also appears paired with itself.

**Columns:** binary variables are 1 when the condition holds for the pair, 0 otherwise. Only `contig` and `comlang_off` are used later.

| Column | Meaning |
|---|---|
| `iso_o`, `iso_d` | ISO3 codes of the origin and destination countries |
| `contig` | **Used.** The two countries share a land border (*contiguity*) |
| `comlang_off` | **Used.** The two countries share an official language |
| `comlang_ethno` | A language is spoken by at least 9% of the population in both countries |
| `colony` | The two countries have ever had a colonial link |
| `comcol` | The two countries had a common colonizer after 1945 |
| `curcol` | The two countries are currently in a colonial relationship |
| `col45` | The two countries were in a colonial relationship after 1945 |
| `smctry` | The two countries were or are the same country (e.g. Czechia and Slovakia) |
| `dist` | Distance between the most populated cities, in km |
| `distcap` | Distance between the capitals, in km |
| `distw` | Average distance between the main cities, weighted by their population, in km |
| `distwces` | Same as `distw`, with a different weighting formula (CES, θ = −1), in km |

In [22]:
# looking for missing values
dist_cepii.isna().sum()

iso_o               0
iso_d               0
contig              0
comlang_off         0
comlang_ethno       0
colony              0
comcol              0
curcol              0
col45               0
smctry              0
dist                0
distcap             0
distw            2215
distwces         2215
dtype: int64

**Missing values:** the two variables used in the project, `contig` and `comlang_off`, have no missing values.